# HW7: Beam Search Decoding - News Headline Generation

In this exercise, you are going to learn and implement decoding techniques for sequence generation. Usually, the sequence is generated word-by-word from a model. In each step, the model predicted the most likely word based on the predicted words in previous steps (this is called auto-regressive decoding).

As such, it is very important how you decide on what to predicted at each step, as it will be conditioned on to predicted all of the following steps. We will implement two of main decoding techniques introduced in the lecture: **Greedy Decoding** and **Beam Search Decoding**. Greedy Decoding immediately chooses the word with best score at each step, while Beam Search Decoding focuses on the sequence that give the best score overall.

To complete this exercise, you will need to complete the methods for decoding for a text generation model trained on [New York Times Comments and Headlines dataset](https://www.kaggle.com/aashita/nyt-comments). The model is trained to predict a headline for the news given seed text. You do not need to train any model model in this exercise as we provide both the pretrained model and dictionary.


## Download model and vocab and setup

In [2]:
!wget -O vocab.txt https://www.dropbox.com/s/ht12ua9vpkep6l8/hw9_vocab.txt?dl=0
!wget -O model.bin https://www.dropbox.com/s/okmri7cnd729rr5/hw9_model.bin?dl=0

--2025-03-09 09:19:16--  https://www.dropbox.com/s/ht12ua9vpkep6l8/hw9_vocab.txt?dl=0
Resolving www.dropbox.com (www.dropbox.com)... 162.125.81.18, 2620:100:6035:18::a27d:5512
Connecting to www.dropbox.com (www.dropbox.com)|162.125.81.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://www.dropbox.com/scl/fi/zlkw3il9cj4c121vtrrxh/hw9_vocab.txt?rlkey=m5gflik1lhvpenwydd3gfwvp7&dl=0 [following]
--2025-03-09 09:19:16--  https://www.dropbox.com/scl/fi/zlkw3il9cj4c121vtrrxh/hw9_vocab.txt?rlkey=m5gflik1lhvpenwydd3gfwvp7&dl=0
Reusing existing connection to www.dropbox.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://uc8489557bfc9603e189d761d002.dl.dropboxusercontent.com/cd/0/inline/ClgAWHRskAcQ12qj-F_QhDg6UKns3pe5YrBS4FUko0ZMRRD08Npj281jwjuYTmwlzi-aAZcHd9eGZVbBzLowCXQgmFCJQ9eG60NOOCexOkL7sFbpcy9yZDYKmxrO4xmGFzs5_DNniSDcmxfJ030NX9j4/file# [following]
--2025-03-09 09:19:17--  https://uc8489557bfc9603e189d761d002.dl.dropboxusercont

In [3]:
import torch
import torch.nn as nn
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import Whitespace

In [4]:
class RNNmodel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, dropout_rate):

        super().__init__()
        self.embedding_dim = embedding_dim

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, 128, num_layers=2,
                     batch_first=True)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(128, vocab_size)

    def forward(self, src):
        embedding = self.embedding(src)
        output,_ = self.rnn(embedding)
        output = self.dropout(output)
        prediction = self.fc2(output)
        return prediction

In [5]:
with open("vocab.txt") as f:
  vocab_file = f.readlines()
embedding_dim = 64
dropout_rate = 0.2

model = RNNmodel(len(vocab_file), embedding_dim, dropout_rate)
model.load_state_dict(torch.load("model.bin",map_location='cpu'))
model.eval()

<ipython-input-5-e56f35018f72>:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("model.bin",map_location='cpu'))


RNNmodel(
  (embedding): Embedding(10054, 64)
  (rnn): LSTM(64, 128, num_layers=2, batch_first=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (fc2): Linear(in_features=128, out_features=10054, bias=True)
)

In [6]:
vocab = [v.strip() for v in vocab_file]
vocab_size = len(vocab)
print(f"Vocab Size: {vocab_size}")
vocab[:10]

Vocab Size: 10054


['<unk>', '<pad>', '<eos>', 'the', 'a', 'to', 'of', 's', 'in', 'for']

In [7]:
stoi = { ch:i for i,ch in enumerate(vocab) }
tokenizer = Tokenizer(WordLevel(stoi, unk_token="<unk>"))
tokenizer.pre_tokenizer = Whitespace()
tokenized_text = tokenizer.encode("the a of to unknowns")
print(tokenized_text)
print(tokenized_text.ids)
print(tokenized_text.tokens)
print(tokenizer.decode(tokenized_text.ids))

Encoding(num_tokens=5, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])
[3, 4, 6, 5, 0]
['the', 'a', 'of', 'to', '<unk>']
the a of to <unk>


## 1. TODO: Greedy decode
Normally, in sequence generation task, the model will continue generating tokens until an end-of-sequence symbol appear or the maximum length is reached. For this task:
- The end-of-sequence symbol is "< eos >" and its index is 2
- Use the maximum generation length of 15

In [8]:
eos_token = '<eos>'
max_gen_length = 15
eos_idx = 2

In [9]:
def greedy_decode(seed_text, tokenizer):
    """Greedy decodes with seed text.

        Args:
        seed_text: The seed string to be used as initial input to the model.
        tokenizer: The tokenizer for converting word to index and back.

        Your code should do the followings:
          1. Convert current_text to sequences of indices
          2. Predict the next token using the model and choose the token with the highest score as output
          3. Append the predicted index to current_text
          4. Loop until completion
          5. Return text prediction and a list of probabilities of each step

        You do not need to stop early when end-of-sequence token is generated and can continue decoding
        until max_gen_length is reached. We can filter the eos token out later.
    """

    encoding = tokenizer.encode(seed_text)
    curr_ids = encoding.ids[:]  # token IDs ของ seed text
    step_probs = []
    with torch.no_grad():
        for _ in range(max_gen_length):
            x = torch.tensor([curr_ids], dtype=torch.long)
            logits = model(x)
            last_logits = logits[0, -1, :]
            probs = torch.softmax(last_logits, dim=0)
            best_prob, best_idx = torch.max(probs, dim=0)
            step_probs.append(best_prob.item())
            curr_ids.append(best_idx.item())
            if best_idx.item() == eos_idx:
                break
    output_text = tokenizer.decode(curr_ids)
    return output_text, step_probs

In [10]:
def clean_output(text, eos_token):
    """Drop eos_token and every words that follow"""
    tokens = text.split()
    if eos_token in tokens:
        tokens = tokens[:tokens.index(eos_token)]
    return " ".join(tokens)

In [11]:
sample_seeds = ["to", "america", "people", "next", "picture", "on"]
for seed in sample_seeds:
    generated_text, step_probs = greedy_decode(seed, tokenizer)
    cleaned_text = clean_output(generated_text, eos_token)
    print(cleaned_text)

to encourage creativity in the new york bill
america s lethal export
people to balloon to make a criminal with a dog with a callous rival
next phenom english clubs 2 call another deal in the same arrivals
picture perfect chapter a spot of view of banning care
on the catwalk in saudi arabia


Your output should be:

*   to encourage creativity in the new york bill
*   america s lethal export
*   people to balloon to make a criminal with a dog with a callous rival
*   next phenom english clubs 2 call another deal in the same arrivals
*   picture perfect chapter a spot of view of banning care  
*   on the catwalk in saudi arabia







## 2. TODO: Beam search decode

Another well-known decoding method is beam search decoding that focuses more on the overall sequence score.

Instead of greedily choosing the token with the highest score for each step, beam search decoding expands all possible next tokens and keeps the __k__ most likely sequence at each step, where __k__ is a user-specified beam size. A sequence score is also calculated according user-specified cal_score() function.
The beam with the highest score after the decoding process is done will be the output.

There are a few things that you need to know before implementing a beam search decoder:
- When the eos token is produced, you can stop expanding that beam
- However, the ended beams must be sorted together with active beams
- The decoding ends when every beams are either ended or reached the maximum length, but for this task, you can continue decoding until the max_gen_len is reached
- We usually work with probability in log scale to avoid numerical underflow. You should use np.log(score) before any calculation
- **As probabilities for some classes will be very small, you must add a very small value to the score before taking log e.g np.log(prob + 0.00000001)**

#### Sequence Score
The naive way to calculate the sequence score is to __multiply every token scores__ together. However, doing so will make the decoder prefer shorter sequence as you multiply the sequence score with a value between \[0,1\] for every tokens in the sequence. Thus, we usually normalize the sequence score with its length by calculating its __geometric mean__ instead.

**You should do this in log scale**

In [12]:
import numpy as np
def cal_score(score_list, length, normalized=False): #cal score for each beam from a list of probs

    eps = 1e-8
    if len(score_list) == 0:
        return 0.0
    log_score = np.sum(np.log(np.array(score_list) + eps))
    if normalized:
        return np.exp(log_score / len(score_list))
    else:
        return np.exp(log_score)

In [13]:
def beam_search_decode(seed_text, max_gen_len, tokenizer, beam_size=5, normalized=False):
    """We will do beam search decoing using seed text in this function.

    Output:
    beams: A list of top k beams after the decoding ended, each beam is a list of
      [seed_text, list of scores, length]

    Your code should do the followings:
    1.Loop until max_gen_len is reached.
    2.During each step, loop thorugh each beam and use it to predict the next word.
      If a beam is already ended, continues without expanding.
    3.Sort all hypotheses according to cal_score().
    4.Keep top k hypotheses to be used at the next step.
    """
    encoding = tokenizer.encode(seed_text)
    init_ids = encoding.ids[:]
    beams = [(init_ids, [], False)]
    with torch.no_grad():
        for _ in range(max_gen_length):
            new_beams = []
            for seq, probs_list, finished in beams:
                if finished:
                    new_beams.append((seq, probs_list, finished))
                    continue
                x = torch.tensor([seq], dtype=torch.long)
                logits = model(x)
                last_logits = logits[0, -1, :]
                curr_probs = torch.softmax(last_logits, dim=0)
                topk_probs, topk_indices = torch.topk(curr_probs, beam_size)
                for prob, idx in zip(topk_probs.tolist(), topk_indices.tolist()):
                    new_seq = seq + [idx]
                    new_probs_list = probs_list + [prob]
                    is_finished = (idx == eos_idx)
                    new_beams.append((new_seq, new_probs_list, is_finished))
            new_beams = sorted(new_beams, key=lambda beam: cal_score(beam[1], normalized), reverse=True)
            beams = new_beams[:beam_size]
            if all(beam[2] for beam in beams):
                break
    return beams

## 3. Generate!
Generate 6 sentences based on the given seed texts.

Decode with the provided seed texts with beam_size 5. Compare the results between greedy, normalized, and unnormalized decoding.

Print the result using greedy decoding and top 2 results each using unnormalized and normalized decoing for each seed text.

Also, print scores of each candidate according to cal_score(). Use normalization for greedy decoding.

In [42]:
sample_seeds = ["to", "america", "people", "next", "picture", "on"]
max_gen_len = 10
beam_size=5

for seed in sample_seeds:
    print("-Greedy-")
    greedy_text, greedy_probs = greedy_decode(seed, tokenizer)
    cleaned_greedy = clean_output(greedy_text, eos_token)
    # สำหรับ Greedy ใช้ normalized score (geometric mean)
    greedy_score = cal_score(greedy_probs, normalized=True)
    print(cleaned_greedy, round(greedy_score, 2))

    beams_unnorm = beam_search_decode(seed, max_gen_length, tokenizer, beam_size=beam_size, normalized=False)
    beams_norm = beam_search_decode(seed, max_gen_length, tokenizer, beam_size=beam_size, normalized=True)

    print("-Unnormalized-")
    for beam in beams_unnorm[:2]:
        seq_ids, probs_list, _ = beam
        decoded = tokenizer.decode(seq_ids)
        # แปลงเป็น Title-case ตามที่โจทย์ต้องการ
        cleaned = clean_output(decoded, eos_token).title()
        score = cal_score(probs_list, normalized=False)
        print(cleaned, round(score, 2))

    print("-Normalized-")
    for beam in beams_norm[:2]:
        seq_ids, probs_list, _ = beam
        decoded = tokenizer.decode(seq_ids)
        cleaned = clean_output(decoded, eos_token).title()
        score = cal_score(probs_list, normalized=True)
        print(cleaned, round(score, 2))
    print("\n")

-Greedy-
to encourage creativity in the new york bill 0.12
-Unnormalized-
To Consult Exploring Recipes For New Jersey 0.0
To Consult Exploring Recipes Up The Pacific Northwest 0.0
-Normalized-
To Consult Exploring Recipes Up The Pacific Northwest 0.22
To Consult Exploring Recipes Up The Least Of The Week 0.19


-Greedy-
america s lethal export 0.35
-Unnormalized-
America S Lethal Export 0.01
America S Desert Aisles 0.01
-Normalized-
America S Lethal Export 0.35
America S Desert Aisles 0.3


-Greedy-
people to balloon to make a criminal with a dog with a callous rival 0.16
-Unnormalized-
People To Balloon For A Criminal 0.0
People To Balloon For A Criminal With Trump 0.0
-Normalized-
People To Balloon For A Criminal With A Second Fiddle 0.16
People To Balloon For A Criminal With Trump 0.16


-Greedy-
next phenom english clubs 2 call another deal in the same arrivals 0.15
-Unnormalized-
Next S Blist Revue 0.0
Next Phenom English Clubs 1 A Chance To Be Back 0.0
-Normalized-
Next S Blist R

Your output should be:


```
-Greedy-
to encourage creativity in the new york bill  0.12
-Unnormalized-
To Consult Exploring Recipes For New Jersey 0.00
To Consult Exploring Recipes Up The Pacific Northwest 0.00
-Normalized-
To Consult Exploring Recipes Up The Pacific Northwest 0.17
To Consult Exploring Recipes Up The Least Of The Week 0.16

-Greedy-
america s lethal export  0.35
-Unnormalized-
America S Lethal Export 0.02
America S Desert Aisles 0.01
-Normalized-
America S Lethal Export 0.25
America S Desert Aisles 0.20

-Greedy-
people to balloon to make a criminal with a dog with a callous rival  0.16
-Unnormalized-
People To Balloon For A Criminal 0.00
People To Balloon For A Criminal With Trump 0.00
-Normalized-
People To Balloon For A Criminal With A Second Fiddle 0.13
People To Balloon For A Criminal With Trump 0.13

-Greedy-
next phenom english clubs 2 call another deal in the same arrivals  0.15
-Unnormalized-
Next S Blist Revue 0.00
Next Phenom English Clubs 1 A Chance To Be Back 0.00
-Normalized-
Next S Blist Revue 0.14
Next Phenom English Clubs 1 A Chance To Be Back 0.14

-Greedy-
picture perfect chapter a spot of view of banning care  0.09
-Unnormalized-
Picture Perfect Use Coffee 0.00
Picture Korean A Bonanza Of Pancakes 0.00
-Normalized-
Picture Korean A Bonanza Of Contemplation Times Of Trump S Son 0.12
Picture Korean A Bonanza Of Pancakes 0.07

-Greedy-
on the catwalk in saudi arabia  0.25
-Unnormalized-
On The Billboard Chart 0.00
On The Catwalk In Saudi Arabia 0.00
-Normalized-
On The Whole30 Diet Vowing To Eat Smarter Carbs To Be 0.27
On The Whole30 Diet Vowing To Eat Smarter Carbs For Because 0.26

```



# Answer Questions in MyCourseVille!

Use the seed word "usa" to answer questions in MCV.

In [44]:
seed = "usa"

# Greedy Decoding
greedy_text, greedy_probs = greedy_decode(seed, tokenizer)
greedy_continuation = clean_output(greedy_text, eos_token)
greedy_score = cal_score(greedy_probs, normalized=True)
print("Greedy Decoding:")
print(greedy_continuation, round(greedy_score, 2))
print()

# Unnormalized Beam Search Decoding
beams_unnorm = beam_search_decode(seed, max_gen_length, tokenizer, beam_size=5, normalized=False)
# เลือก beam ที่ดีที่สุด
unnorm_continuation = clean_output(tokenizer.decode(beams_unnorm[0][0]), eos_token)
unnorm_score = cal_score(beams_unnorm[0][1], normalized=False)
print("Unnormalized Beam Search Decoding:")
print(unnorm_continuation.title(), round(unnorm_score, 2))
print()

# Normalized Beam Search Decoding
beams_norm = beam_search_decode(seed, max_gen_length, tokenizer, beam_size=5, normalized=True)
norm_continuation = clean_output(tokenizer.decode(beams_norm[0][0]), eos_token)
norm_score = cal_score(beams_norm[0][1], normalized=True)
print("Normalized Beam Search Decoding:")
print(norm_continuation.title(), round(norm_score, 2))


Greedy Decoding:
usa s duty to investigate 0.15

Unnormalized Beam Search Decoding:
Usa S Duty To Investigate 0.0

Normalized Beam Search Decoding:
Usa S Bleak Season 1 Episode 2 Darkness Descends 0.23
